# Tugas Minggu 11 - Association Rule (Praktikum Data Mining)

**Nama  :** Muhammad Ikhwan Fitoriqillah  
**Mata Kuliah  :** Data Mining  
**Dataset  :** transaction.csv  
**Tools  :** Python (pandas + mlxtend)

Tugas:
1. Load dataset `transaction.csv` dan tampilkan.
2. Ambil data hanya untuk negara **Portugal**.
3. Buat tabel transaksi: 1 InvoiceNo = 1 transaksi, isi nya StockCode.
4. Cari association rule dengan **min_support = 0.2** dan **min_confidence = 0.7**.


## 0) Import library yang dipakai

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

## 1) Load dataset transaction.csv dan tampilkan

In [2]:
dataset = pd.read_csv("transaction.csv")
print("Jumlah baris :", dataset.shape[0])
print("Jumlah kolom :", dataset.shape[1])
print()
print("Tampilan dataset:")
dataset

Jumlah baris : 10546
Jumlah kolom : 6

Tampilan dataset:


,InvoiceNo,StockCode,Qty,InvoiceDate,CustomerID,Country
0,537626,22725,830,12/7/2010 14:57,12347,Iceland
1,537626,22729,948,12/7/2010 14:57,12347,Iceland
2,537626,22195,695,12/7/2010 14:57,12347,Iceland
3,542237,22725,636,1/26/2011 14:30,12347,Iceland
4,542237,22729,536,1/26/2011 14:30,12347,Iceland
...,...,...,...,...,...,...
10541,543911,21700,455,2/14/2011 12:46,17829,United Arab Emirates
10542,543911,22111,578,2/14/2011 12:46,17829,United Arab Emirates
10543,543911,22112,163,2/14/2011 12:46,17829,United Arab Emirates
10544,564428,23296,545,8/25/2011 11:27,17844,Canada


## 2) Ambil data untuk negara Portugal

In [3]:
data = dataset[dataset["Country"] == "Portugal"]
print("Jumlah baris data Portugal :", data.shape[0])
print("Jumlah InvoiceNo unik       :", data["InvoiceNo"].nunique())
print("Jumlah StockCode unik       :", data["StockCode"].nunique())
print()
print("Tampilan data Portugal:")
data

Jumlah baris data Portugal : 367
Jumlah InvoiceNo unik       : 43
Jumlah StockCode unik       : 140

Tampilan data Portugal:


,InvoiceNo,StockCode,Qty,InvoiceDate,CustomerID,Country
101,541430,22195,649,1/18/2011 9:50,12356,Portugal
102,541430,22435,460,1/18/2011 9:50,12356,Portugal
103,541430,84378,304,1/18/2011 9:50,12356,Portugal
104,541430,22646,896,1/18/2011 9:50,12356,Portugal
105,541430,84987,157,1/18/2011 9:50,12356,Portugal
...,...,...,...,...,...,...
7596,547444,22745,292,3/23/2011 10:55,12811,Portugal
7597,547444,22747,133,3/23/2011 10:55,12811,Portugal
7598,547444,22746,835,3/23/2011 10:55,12811,Portugal
7599,547444,23176,878,3/23/2011 10:55,12811,Portugal


## 3) Buat tabel transaksi (1 InvoiceNo = 1 transaksi)

Caranya mirip seperti contoh di slide: `groupby([InvoiceNo, StockCode])[Qty].sum()`, lalu `unstack`, isi NaN dengan 0, lalu nilai apapun yang lebih dari 0 diubah jadi 1 (one-hot).

In [4]:
transaksi = data.groupby(["InvoiceNo", "StockCode"])["Qty"].sum()
transaksi = transaksi.unstack().reset_index().fillna(0).set_index("InvoiceNo")
transaksi[transaksi > 0] = 1
transaksi = transaksi.astype(bool)  # mlxtend butuh tipe bool / 0-1
print("Bentuk tabel transaksi (baris=transaksi, kolom=StockCode) :", transaksi.shape)
print()
print("Tabel Transaksi:")
transaksi

Bentuk tabel transaksi (baris=transaksi, kolom=StockCode) : (43, 140)

Tabel Transaksi:


StockCode,16161,20713,20718,20723,20977,20981,21035,21124,21154,21164,...,47599,48184,82484,84077,84279,84378,84380,84509,84987,84988
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
537246,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
537818,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,False,False,False
537915,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
538311,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
539353,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
540519,False,False,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
540546,False,False,False,False,True,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
541430,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,True,False,True,False
542147,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False


## 4) Cari association rule (min_support = 0.2 dan min_confidence = 0.7)

In [5]:
frequent_itemsets = apriori(transaksi, min_support=0.2, use_colnames=True)
print("Jumlah frequent itemsets :", len(frequent_itemsets))
print()
print("Frequent Itemsets:")
frequent_itemsets.sort_values("support", ascending=False)

Jumlah frequent itemsets : 7

Frequent Itemsets:


,support,itemsets
2,0.255814,frozenset({22411})
0,0.232558,frozenset({21928})
1,0.209302,frozenset({21929})
3,0.209302,"frozenset({21928, 21929})"
4,0.209302,"frozenset({21928, 22411})"
5,0.209302,"frozenset({21929, 22411})"
6,0.209302,"frozenset({21928, 21929, 22411})"


In [6]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)
print("Jumlah association rules :", len(rules))
print()
print("Association Rules:")
rules[["antecedents", "consequents", "support", "confidence", "lift"]].sort_values("confidence", ascending=False)

Jumlah association rules : 12

Association Rules:


,antecedents,consequents,support,confidence,lift
1,frozenset({21929}),frozenset({21928}),0.209302,1.000000,4.300000
7,"frozenset({21928, 22411})",frozenset({21929}),0.209302,1.000000,4.777778
6,"frozenset({21928, 21929})",frozenset({22411}),0.209302,1.000000,3.909091
4,frozenset({21929}),frozenset({22411}),0.209302,1.000000,3.909091
10,frozenset({21929}),"frozenset({21928, 22411})",0.209302,1.000000,4.777778
8,"frozenset({21929, 22411})",frozenset({21928}),0.209302,1.000000,4.300000
0,frozenset({21928}),frozenset({21929}),0.209302,0.900000,4.300000
2,frozenset({21928}),frozenset({22411}),0.209302,0.900000,3.518182
9,frozenset({21928}),"frozenset({21929, 22411})",0.209302,0.900000,4.300000
3,frozenset({22411}),frozenset({21928}),0.209302,0.818182,3.518182


### Kesimpulan singkat

- Dataset awal punya banyak negara, tetapi yang dipakai cuma transaksi dari **Portugal**.
- Setelah dijadikan tabel transaksi (1 InvoiceNo = 1 baris, kolomnya StockCode), Apriori dijalankan dengan minimum support 0.2.
- Dari frequent itemsets di atas, dihasilkan association rule dengan minimum confidence 0.7.
- Rule yang muncul menunjukkan StockCode mana yang sering dibeli bersamaan oleh pelanggan di Portugal.
